# The Climate is Changing: How is Our Sentiment?

## Exploring Changing Sentiment in Climate Articles: 2013 to 2023

#### This is a notebook for initial exploration of the corpus.

## Corpus Description

9 scientific articles are drawn from PeerJ's publications in the decade spanning 2013 to 2023 (3 from 2013, 3 from 2018, and 3 from 2023). Articles are filtered so as to have 50 or more citations, and are sorted by PeerJ's determination of relevance to the query "climate." The top 3 most relevant articles, after meeting these described criteria, have then been chosen to comprise my corpus.

## Imports and Set-Up

In [ ]:
# General imports
import numpy as np 
import pandas as pd 
import os
from collections import Counter
import re

# XML data
from bs4 import BeautifulSoup
import xml.etree.ElementTree as ET
import lxml

# Plotting
import plotly.express as px
import plotly.io as pio
import matplotlib.pyplot as plt

# Gensim and TSNE
from gensim.corpora import Dictionary
from gensim.models import word2vec

# SKLearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.manifold import TSNE as tsne

# Setting up input data
files = []
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        files.append(os.path.join(dirname, filename))

print(f"There are {len(files)} documents in the corpus.")

In [ ]:
pio.renderers.default = 'iframe'
import warnings; warnings.filterwarnings('ignore', category=FutureWarning)
import gensim; gensim.__version__

## EDA

In [ ]:
# From https://www.zyte.com/learn/a-practical-guide-to-xml-parsing-with-python/

# Load XML data from a file
print("First File")
tree = ET.parse(files[0])
root = tree.getroot()

# Print the root element
print(f"Root: {root.tag}")

# Access child elements
for child in root:
    print(f"Child Tag: {child.tag} \nChild Text: {child.text}")

for grandchild in child:
    print(f"Grandchild Tag: {grandchild.tag} \nGrandchild Text: {grandchild.text}")

In [ ]:
# From Claude:
# Use a namespace-aware find (JATS has no default namespace, so this is straightforward)
for sec in root.iter('sec'):
    title = sec.find('title')
    if title is not None:
        print(f"Section: {title.text}")
    for p in sec.findall('p'):
        print(f"  Paragraph: {p.text[:80] if p.text else '(no direct text)'}")

### EDA: Average, Max, Min Length of Docs in Corpus

In [ ]:
# Create an array of counters for each document
counters = np.zeros(len(files))

# For each doc in the corpus:
for i, file in enumerate(files):
    # Get the tree and the root
    tree = ET.parse(file)
    root = tree.getroot()
    # Iterate through sections, titles, and paragraphs
    for sec in root.iter('sec'):
        title = sec.find('title')
        if title is not None:
            # Add to the appropriate counter
            counters[i] += len(title.text)
        for p in sec.findall('p'):
            counters[i] += len(p.text) if p.text else 0

# Average length of documents in corpus
avg = np.sum(counters)/len(counters)
print(f"The average length of each document in the corpus is {avg:.0f} characters long.")

# Mininmum length
min = np.min(counters)
print(f"The longest document in the corpus is {min:.0f} characters long.")

# Maxinmum length
max = np.max(counters)
print(f"The longest document in the corpus is {max:.0f} characters long.")

### EDA: Exploring OHCO Structure

After rigorous examination of each corpus, I have determined that the following OHCO encompasses the meaningful prose of each article down to the sentence level:

- Article
- Body
- Sec
- P

In [ ]:
# Function to define OHCO structure
# Debugging conducted by Claude upon original creation of preliminary function
def get_ohco(file):
    # Define tree and root
    tree = ET.parse(file)
    root = tree.getroot()
    # Array to hold OHCO
    OHCO = []
    # Recursive helper function
    def make_loop(node, depth=0):
        OHCO.append((node.tag))
        for child in node:
            make_loop(child, depth + 1)
    make_loop(root)
    return OHCO

OHCO = get_ohco(files[0])
# View of a sample file's OHCO 
OHCO[:10]

In [ ]:
OHCO = ['article', 'front', 'body', 'sec', 'p']

## TOKENS Table

In [ ]:
def tokens(file):
    """
    Takes an XML file as input and converts it to a dataframe of strings in the document split by line
    """
    # Convert file to a df
    docs = pd.DataFrame(dict(doc_str=open(file, 'r').read().split('><')))
    
    # Clean up strings and remove blank lines
    docs.doc_str = docs.doc_str.str.replace(r"\n+", " ", regex=True)
    docs = docs[docs.doc_str != '']
    
    # Remove back section so citations do not impurify the analysis of a given article on its own
    end = docs[docs.doc_str.str.contains("<back>")].index.values[0]
    docs = docs[(docs.index < end)]
    
    # Create index
    docs = docs.reset_index(drop=True)
    docs.index.name = 'doc_id'
    
    return docs

# Store dataframes in a dict
all_docs = {}

# Create dataframes for each file
for i, file in enumerate(files):
    docs = tokens(file)
    docs = docs.astype(str)
    all_docs[i] = docs

all_docs[0]

In [ ]:
# Vectorize corpus
def make_vector(doc):
    # Create engine, model
    engine = CountVectorizer()
    model = engine.fit_transform(doc['doc_str']) # Claude suggested this, passes as series so it's accepted by CountVectorizer()
    
    # Extract count matrix
    V = engine.get_feature_names_out()

    # Data frame
    X = pd.DataFrame(model.toarray(), columns=V)
    X.index.name = 'doc_id'

    return X

vectorized = {}

# Create dataframes for each file
for i, doc in all_docs.items():
    vector = make_vector(doc)
    vectorized[i] = vector

vectorized[0]

### Extract Vocabulary

In [ ]:
def extract_vocab(vector):
    V = vector.sum().to_frame('n')
    V.index.name = 'term_str'
    # Show stats
    V.sort_values('n').tail(20).plot.barh(figsize=(10,10))
    plt.show()
    return V

Vs = {}

# Create dataframes for each file
for i, vector in vectorized.items():
    V = extract_vocab(vector)
    Vs[i] = V

Vs[0]

**Observations:**

Many of the top words seem to come from the XML file structure itself and from formatting keywords, but some science-leaning words do emerge: barley, yeast, species, soil, etc. I wonder if it will be necessary to remove some of these structure and formatting words manually as stopwords.

### Incorporating OHCO Structure

Reminder: OHCO structure = 'article', 'front', 'body', 'sec', 'p'

In [ ]:
article_pat = "<article "
article_lines = all_docs[0].doc_str.str.match(chap_pat, case=False) # Returns a truth vector
# line_str replaced with doc_str

## Gensim, Word2Vec, and TSNE Spaces

In [ ]:
# Define a function to convert into Gensim, apply word2vec, and create TSNE spaces
def convert_gensim(corpus):
    OHCO = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']
    BAG = OHCO[:6] 
    TOKENS = corpus.set_index(OHCO).dropna()
    docs = TOKENS.groupby(BAG).term_str.apply(list).tolist()
    # word2vec parameters
    w2v_params = dict(
        window = 2, 
        vector_size = 256,
        min_count = 50,
        workers = 1,
        seed = 42
    )
    model = word2vec.Word2Vec(docs, **w2v_params)
    vecs = model.wv.vectors
    WV = pd.DataFrame(vecs, index=model.wv.index_to_key)
    # TSNE
    tsne_params = dict(
    perplexity=20, 
    n_components=2, 
    init='pca', 
    max_iter=1000, 
    random_state=42)
    tsne_engine = tsne(**tsne_params)
    TSNE = pd.DataFrame(
        tsne_engine.fit_transform(WV), 
        columns=['x','y'], 
        index=WV.index)
    return docs, WV, TSNE, model

docs, WV, TSNE, model = convert_gensim(Vs[0])